# S7 — 오염 시행 제거 효과: RAM 절약형 LOSO

VS Code에서 이 `.ipynb`를 열고 **Google Colab GPU 런타임**을 커널로 선택해 실행합니다. 새 Google 계정의 `MyDrive/BCI_Research`를 사용합니다. 로컬 Python 커널에서는 실행되지 않습니다.

HDF5에서 필요한 시행만 읽어 Colab RAM 12.67GB 환경에서도 전체 피험자를 메모리에 올리지 않습니다. 기존 Drive CSV의 완료된 피험자는 건너뜁니다.

외부 테스트 피험자는 학습 중 조기 종료에 사용하지 않습니다. 동일한 깨끗한 테스트 시행에서 `dirty_train`과 `clean_train`을 비교합니다. 각 조합은 완료된 피험자를 건너뛰므로 중단 후 해당 셀을 다시 실행할 수 있습니다. 총 18개 조합(3모델 × 2조건 × 3시드)은 GPU에서도 오래 걸립니다.

**한계:** 이 실험은 저장된 HDF5에서 오염 시행을 배제하고 깨끗한 시행으로 EEG 정규화를 다시 계산합니다. 원본 연속 신호에서 ICA를 다시 학습하는 실험은 별개입니다.


In [ ]:
# 1. 새 계정의 Drive 연결 및 GPU 확인
import os, torch
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'GPU 런타임을 선택하세요: 런타임 → 런타임 유형 변경 → GPU'
DRIVE_ROOT = Path('/content/drive/MyDrive/BCI_Research')
assert DRIVE_ROOT.is_dir(), f'새 계정 Drive에 폴더가 없습니다: {DRIVE_ROOT}'
print('GPU:', torch.cuda.get_device_name(0))
print('Drive:', DRIVE_ROOT)


In [ ]:
# 2. Install verified code and exclusion manifest
import base64, hashlib
CODE_DIR = Path("/content/mi_bci_rigorous")
CODE_DIR.mkdir(exist_ok=True)
(CODE_DIR / 'ablation_study_v5.py').write_bytes(base64.b64decode('...'))
assert hashlib.sha256((CODE_DIR / 'ablation_study_v5.py').read_bytes()).hexdigest() == 'edb9ad5dca471de6e4d90738699e6d9f67428f2ca511d93988761d22d02d5761'
(CODE_DIR / 'rigorous_loso.py').write_bytes(base64.b64decode('...'))
assert hashlib.sha256((CODE_DIR / 'rigorous_loso.py').read_bytes()).hexdigest() == 'cad5e0de25703806a9225635806676add5b9376aad4fc2e5f621138ae92cb40b'
(CODE_DIR / 'analyze_rigorous_loso.py').write_bytes(base64.b64decode('IiIiQW5hbHl6ZSBjb21wbGV0ZSBwYWlyZWQgcmlnb3JvdXNfbG9zbyBydW5zIHdpdGhvdXQgdHJlYXRpbmcgc2VlZHMgYXMgc3ViamVjdHMuIiIiCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHBhbmRhcyBhcyBwZApmcm9tIHNjaXB5LnN0YXRzIGltcG9ydCB3aWxjb3hvbgoKUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnQucGFyZW50CkRFRkFVTFRfUkVTVUxUUyA9IFJPT1QgLyAiQkNJX1Jlc2VhcmNoL3Jlc3VsdHMvcmlnb3JvdXNfbG9zbyIKQ09ORElUSU9OUyA9ICgiZGlydHlfdHJhaW4iLCAiY2xlYW5fdHJhaW4iKQpNT0RFTFMgPSAoImVlZ19vbmx5IiwgImVtZ19vbmx5IiwgImZ1c2lvbiIpCk1FVFJJQ1MgPSAoImFjY3VyYWN5IiwgImthcHBhIikKCgpkZWYgbG9hZF9jb21wbGV0ZShvdXRfZGlyLCBzZWVkcywgZXhwZWN0ZWRfc3ViamVjdHM9NDkpOgogICAgZnJhbWVzID0gW10KICAgIGZvciBjb25kaXRpb24gaW4gQ09ORElUSU9OUzoKICAgICAgICBmb3IgbW9kZWwgaW4gTU9ERUxTOgogICAgICAgICAgICBmb3Igc2VlZCBpbiBzZWVkczoKICAgICAgICAgICAgICAgIHBhdGggPSBvdXRfZGlyIC8gZiJ7Y29uZGl0aW9ufV97bW9kZWx9X3NlZWR7c2VlZH0uY3N2IgogICAgICAgICAgICAgICAgaWYgbm90IHBhdGguaXNfZmlsZSgpOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiUmVxdWlyZWQgcnVuIGlzIG1pc3Npbmc6IHtwYXRofSIpCiAgICAgICAgICAgICAgICBkZiA9IHBkLnJlYWRfY3N2KHBhdGgpCiAgICAgICAgICAgICAgICBpZiAobGVuKGRmKSAhPSBleHBlY3RlZF9zdWJqZWN0cyBvciBkZi5zaWQubnVuaXF1ZSgpICE9IGV4cGVjdGVkX3N1YmplY3RzCiAgICAgICAgICAgICAgICAgICAgICAgIG9yIHNldChkZi5jb25kaXRpb24pICE9IHtjb25kaXRpb259CiAgICAgICAgICAgICAgICAgICAgICAgIG9yIHNldChkZi5tb2RlbF90eXBlKSAhPSB7bW9kZWx9CiAgICAgICAgICAgICAgICAgICAgICAgIG9yIHNldChkZi5zZWVkKSAhPSB7c2VlZH0pOgogICAgICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJJbmNvbXBsZXRlIG9yIGluY29uc2lzdGVudCBydW46IHtwYXRofSIpCiAgICAgICAgICAgICAgICBmcmFtZXMuYXBwZW5kKGRmKQogICAgYWxsX3Jvd3MgPSBwZC5jb25jYXQoZnJhbWVzLCBpZ25vcmVfaW5kZXg9VHJ1ZSkKICAgIGlkcyA9IHt0dXBsZShzb3J0ZWQoZi5zaWQpKSBmb3IgZiBpbiBmcmFtZXN9CiAgICBpZiBsZW4oaWRzKSAhPSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkNvbmRpdGlvbnMgZG8gbm90IGNvbnRhaW4gdGhlIHNhbWUgdGVzdCBzdWJqZWN0cyIpCiAgICBncm91cGVkID0gYWxsX3Jvd3MuZ3JvdXBieShbInNlZWQiLCAic2lkIl0pCiAgICBpZiBncm91cGVkLnZhbGlkYXRpb25fc2lkLm51bmlxdWUoKS5tYXgoKSAhPSAxOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIlBhaXJlZCBydW5zIHNlbGVjdGVkIGRpZmZlcmVudCB2YWxpZGF0aW9uIHN1YmplY3RzIikKICAgIGlmIGdyb3VwZWQubl90cmlhbHMubnVuaXF1ZSgpLm1heCgpICE9IDE6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiUGFpcmVkIHJ1bnMgdXNlZCBkaWZmZXJlbnQgY2xlYW4gdGVzdC10cmlhbCBjb3VudHMiKQogICAgcmV0dXJuIGFsbF9yb3dzCgoKZGVmIHBhaXJlZF9yZXN1bHQoZGYsIG51bWVyYXRvciwgZGVub21pbmF0b3IsIG1ldHJpYywgYm9vdHN0cmFwPTEwMDAwKToKICAgICIiIkF2ZXJhZ2UgZWFjaCBzdWJqZWN0IGFjcm9zcyBzZWVkcywgdGhlbiB0ZXN0IHBhaXJlZCBzdWJqZWN0IGRpZmZlcmVuY2VzLiIiIgogICAgcGVyX3N1YmplY3QgPSBkZi5ncm91cGJ5KFsiY29uZGl0aW9uIiwgIm1vZGVsX3R5cGUiLCAic2lkIl0pW21ldHJpY10ubWVhbigpCiAgICBhID0gcGVyX3N1YmplY3QueHMobnVtZXJhdG9yWzoyXSkuc29ydF9pbmRleCgpCiAgICBiID0gcGVyX3N1YmplY3QueHMoZGVub21pbmF0b3JbOjJdKS5zb3J0X2luZGV4KCkKICAgIGlmIG5vdCBhLmluZGV4LmVxdWFscyhiLmluZGV4KToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJTdWJqZWN0IHBhaXJpbmcgZmFpbGVkIikKICAgIGRlbHRhID0gKGEgLSBiKS50b19udW1weShkdHlwZT1mbG9hdCkKICAgIHAgPSBmbG9hdCh3aWxjb3hvbihkZWx0YSkucHZhbHVlKSBpZiBucC5hbnkoZGVsdGEgIT0gMCkgZWxzZSAxLjAKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZyg0MikKICAgIGRyYXdzID0gcm5nLmNob2ljZShkZWx0YSwgc2l6ZT0oYm9vdHN0cmFwLCBsZW4oZGVsdGEpKSwgcmVwbGFjZT1UcnVlKS5tZWFuKGF4aXM9MSkKICAgIGxvdywgaGlnaCA9IG5wLnF1YW50aWxlKGRyYXdzLCBbMC4wMjUsIDAuOTc1XSkKICAgIHJldHVybiB7Im5fc3ViamVjdHMiOiBsZW4oZGVsdGEpLCAibWVhbl9hIjogZmxvYXQoYS5tZWFuKCkpLAogICAgICAgICAgICAibWVhbl9iIjogZmxvYXQoYi5tZWFuKCkpLCAibWVhbl9wYWlyZWRfZGVsdGEiOiBmbG9hdChkZWx0YS5tZWFuKCkpLAogICAgICAgICAgICAiYm9vdHN0cmFwXzk1Y2kiOiBbZmxvYXQobG93KSwgZmxvYXQoaGlnaCldLCAid2lsY294b25fcCI6IHAsCiAgICAgICAgICAgICJwb3NpdGl2ZV9zdWJqZWN0cyI6IGludCgoZGVsdGEgPiAwKS5zdW0oKSksCiAgICAgICAgICAgICJuZWdhdGl2ZV9zdWJqZWN0cyI6IGludCgoZGVsdGEgPCAwKS5zdW0oKSksCiAgICAgICAgICAgICJ6ZXJvX3N1YmplY3RzIjogaW50KChkZWx0YSA9PSAwKS5zdW0oKSl9CgoKZGVmIG1haW4oKToKICAgIHBhcnNlciA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dF9kaXIiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9REVGQVVMVF9SRVNVTFRTKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1zZWVkcyIsIHR5cGU9aW50LCBuYXJncz0iKyIsIGRlZmF1bHQ9WzQyLCAxMzM3LCAyMDI0XSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tYm9vdHN0cmFwIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTAwMDApCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgaWYgbGVuKHNldChhcmdzLnNlZWRzKSkgIT0gbGVuKGFyZ3Muc2VlZHMpIG9yIGFyZ3MuYm9vdHN0cmFwIDwgMTAwOgogICAgICAgIHBhcnNlci5lcnJvcigiU2VlZHMgbXVzdCBiZSB1bmlxdWUgYW5kIGJvb3RzdHJhcCBtdXN0IGJlIGF0IGxlYXN0IDEwMCIpCiAgICBkZiA9IGxvYWRfY29tcGxldGUoYXJncy5vdXRfZGlyLCBhcmdzLnNlZWRzKQoKICAgIGFuYWx5c2VzID0ge30KICAgIGZvciBtZXRyaWMgaW4gTUVUUklDUzoKICAgICAgICBmb3IgbW9kZWwgaW4gTU9ERUxTOgogICAgICAgICAgICBrZXkgPSBmInRyYWluaW5nX2NsZWFudXBfe21vZGVsfV97bWV0cmljfSIKICAgICAgICAgICAgYW5hbHlzZXNba2V5XSA9IHBhaXJlZF9yZXN1bHQoCiAgICAgICAgICAgICAgICBkZiwgKCJjbGVhbl90cmFpbiIsIG1vZGVsKSwgKCJkaXJ0eV90cmFpbiIsIG1vZGVsKSwKICAgICAgICAgICAgICAgIG1ldHJpYywgYXJncy5ib290c3RyYXApCiAgICAgICAgZm9yIGNvbmRpdGlvbiBpbiBDT05ESVRJT05TOgogICAgICAgICAgICBrZXkgPSBmImZ1c2lvbl9nYWluX3tjb25kaXRpb259X3ttZXRyaWN9IgogICAgICAgICAgICBhbmFseXNlc1trZXldID0gcGFpcmVkX3Jlc3VsdCgKICAgICAgICAgICAgICAgIGRmLCAoY29uZGl0aW9uLCAiZnVzaW9uIiksIChjb25kaXRpb24sICJlZWdfb25seSIpLAogICAgICAgICAgICAgICAgbWV0cmljLCBhcmdzLmJvb3RzdHJhcCkKCiAgICAjIERpZmZlcmVuY2UgaW4gZnVzaW9uIGFkdmFudGFnZSBhdHRyaWJ1dGFibGUgdG8gdHJhaW5pbmctdHJpYWwgY2xlYW51cC4KICAgIGZvciBtZXRyaWMgaW4gTUVUUklDUzoKICAgICAgICBwaXZvdCA9IGRmLmdyb3VwYnkoWyJzaWQiLCAiY29uZGl0aW9uIiwgIm1vZGVsX3R5cGUiXSlbbWV0cmljXS5tZWFuKCkudW5zdGFjaygKICAgICAgICAgICAgWyJjb25kaXRpb24iLCAibW9kZWxfdHlwZSJdKQogICAgICAgIGRpcnR5ID0gcGl2b3RbKCJkaXJ0eV90cmFpbiIsICJmdXNpb24iKV0gLSBwaXZvdFsoImRpcnR5X3RyYWluIiwgImVlZ19vbmx5IildCiAgICAgICAgY2xlYW4gPSBwaXZvdFsoImNsZWFuX3RyYWluIiwgImZ1c2lvbiIpXSAtIHBpdm90WygiY2xlYW5fdHJhaW4iLCAiZWVnX29ubHkiKV0KICAgICAgICBkZWx0YSA9IChjbGVhbiAtIGRpcnR5KS50b19udW1weShkdHlwZT1mbG9hdCkKICAgICAgICBybmcgPSBucC5yYW5kb20uZGVmYXVsdF9ybmcoNDIpCiAgICAgICAgZHJhd3MgPSBybmcuY2hvaWNlKGRlbHRhLCBzaXplPShhcmdzLmJvb3RzdHJhcCwgbGVuKGRlbHRhKSksIHJlcGxhY2U9VHJ1ZSkubWVhbihheGlzPTEpCiAgICAgICAgYW5hbHlzZXNbZiJkaWZmZXJlbmNlX2luX2Z1c2lvbl9nYWluX3ttZXRyaWN9Il0gPSB7CiAgICAgICAgICAgICJuX3N1YmplY3RzIjogbGVuKGRlbHRhKSwgImRpcnR5X2dhaW5fbWVhbiI6IGZsb2F0KGRpcnR5Lm1lYW4oKSksCiAgICAgICAgICAgICJjbGVhbl9nYWluX21lYW4iOiBmbG9hdChjbGVhbi5tZWFuKCkpLAogICAgICAgICAgICAiY2hhbmdlX21lYW4iOiBmbG9hdChkZWx0YS5tZWFuKCkpLAogICAgICAgICAgICAiYm9vdHN0cmFwXzk1Y2kiOiBbZmxvYXQoeCkgZm9yIHggaW4gbnAucXVhbnRpbGUoZHJhd3MsIFswLjAyNSwgMC45NzVdKV0sCiAgICAgICAgICAgICJ3aWxjb3hvbl9wIjogZmxvYXQod2lsY294b24oZGVsdGEpLnB2YWx1ZSkgaWYgbnAuYW55KGRlbHRhICE9IDApIGVsc2UgMS4wLAogICAgICAgIH0KCiAgICByZXN1bHQgPSB7InByb3RvY29sIjogInN1YmplY3QtZGlzam9pbnQgdmFsaWRhdGlvbjsgY2xlYW4gb3V0ZXIgdGVzdCBpbiBib3RoIGNvbmRpdGlvbnMiLAogICAgICAgICAgICAgICJzZWVkcyI6IGFyZ3Muc2VlZHMsICJhbmFseXNlcyI6IGFuYWx5c2VzfQogICAgb3V0cHV0ID0gYXJncy5vdXRfZGlyIC8gInBhaXJlZF9hbmFseXNpcy5qc29uIgogICAgb3V0cHV0LndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXN1bHQsIGluZGVudD0yKSkKICAgIHByaW50KGYiU2F2ZWQge291dHB1dH0iKQogICAgZm9yIG5hbWUsIHZhbHVlIGluIGFuYWx5c2VzLml0ZW1zKCk6CiAgICAgICAgZWZmZWN0ID0gdmFsdWUuZ2V0KCJtZWFuX3BhaXJlZF9kZWx0YSIsIHZhbHVlLmdldCgiY2hhbmdlX21lYW4iKSkKICAgICAgICBwcmludChmIntuYW1lfTogZGVsdGE9e2VmZmVjdDorLjRmfSwgOTUlIENJPXt2YWx1ZVsnYm9vdHN0cmFwXzk1Y2knXX0sICIKICAgICAgICAgICAgICBmInA9e3ZhbHVlWyd3aWxjb3hvbl9wJ106LjRnfSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='))
assert hashlib.sha256((CODE_DIR / 'analyze_rigorous_loso.py').read_bytes()).hexdigest() == 'b20e65f1cc4e08797af7c6f8a10d7656c6a01b42c5e6b4e89724eda113ebfe4a'
(CODE_DIR / 'excluded_epochs.json').write_bytes(base64.b64decode('...'))
assert hashlib.sha256((CODE_DIR / 'excluded_epochs.json').read_bytes()).hexdigest() == 'c59d37613123ae1408c9d9b1cd260f0a6a665aeb335c4051ab024955c2487075'
print("Verified code and exclusions:", CODE_DIR)


In [ ]:
# 3. 입력 검증 및 HDF5를 로컬 런타임 디스크에 복사
import json, shutil, subprocess, sys, time
manifest = json.loads((CODE_DIR / 'excluded_epochs.json').read_text())
assert len(manifest) == 51 and sum(len(v['excl_idx']) for v in manifest.values()) == 753
source = DRIVE_ROOT / 'preprocessed/member_A'
assert source.is_dir(), f'HDF5 폴더가 없습니다: {source}'
needed = [source / f'sub-{int(s):02d}_member_A.h5' for s in sorted(manifest, key=int)]
missing = [p.name for p in needed if not p.is_file()]
assert not missing, f'HDF5 파일 누락: {missing}'
bytes_needed = sum(p.stat().st_size for p in needed)
local = Path('/content/bci_data')
local.mkdir(exist_ok=True)
free = shutil.disk_usage('/content').free
remaining_bytes = sum(p.stat().st_size for p in needed if not (local / p.name).is_file() or (local / p.name).stat().st_size != p.stat().st_size)
assert free >= remaining_bytes + 4_000_000_000, f'로컬 디스크 부족: 추가 필요 {remaining_bytes/1e9:.1f}GB, 여유 {free/1e9:.1f}GB'
for i, p in enumerate(needed, 1):
    target = local / p.name
    if not target.is_file() or target.stat().st_size != p.stat().st_size:
        shutil.copy2(p, target)
    if i % 10 == 0 or i == len(needed):
        print(f'{i}/{len(needed)} 파일 확인')
import h5py
for sid in sorted(manifest, key=int):
    with h5py.File(local / f'sub-{int(sid):02d}_member_A.h5') as f:
        n = manifest[sid]['n_total']
        assert f['labels'].shape[0] == n, f'라벨 수 불일치: s{int(sid):02d}'
        assert f['eeg/epochs'].shape == (n, 64, 2304), f'EEG shape 불일치: s{int(sid):02d}'
        assert f['emg/epochs'].shape == (n, 4, 2304), f'sEMG shape 불일치: s{int(sid):02d}'
OUT_DIR = DRIVE_ROOT / 'results/rigorous_loso'
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = local
EXCL_JSON = CODE_DIR / 'excluded_epochs.json'
print('준비 완료:', len(needed), 'subjects; output:', OUT_DIR)

# Stream child stdout/stderr into the notebook and retain the actual error.
from collections import deque
def run_training(cmd):
    tail = deque(maxlen=40)
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            tail.append(line.rstrip())
        return_code = process.wait()
    if return_code:
        raise RuntimeError(f'Training process exited with code {return_code}. Last output:\n' + '\n'.join(tail))


## 재학습

각 실행은 시작 시 `완료 N/49`, 피험자 시작·완료 시 `[N/49]`, 학습 중 5 epoch마다 검증 F1을 출력합니다. 바로 아래 진행 현황 셀을 다시 실행하면 18개 조합의 완료 수와 최근 완료 피험자를 확인할 수 있습니다. 끊긴 조합은 동일한 셀을 다시 실행하면 완료된 피험자를 건너뜁니다. 재개 시 준비 셀 1–3을 먼저 다시 실행하세요. 출력 CSV와 설정 파일은 Drive에 저장됩니다.


In [ ]:
# Saved progress for all 18 combinations. Re-run any time.
import csv
def show_progress():
    print(f'{"seed":>5}  {"model":<8}  {"condition":<11}  {"done":>5}  {"last":>7}')
    grand_total = 0
    for seed in (42, 1337, 2024):
        for model in ('eeg_only', 'emg_only', 'fusion'):
            for condition in ('dirty_train', 'clean_train'):
                path = OUT_DIR / f'{condition}_{model}_seed{seed}.csv'
                if path.is_file():
                    with path.open(newline='') as stream:
                        rows = list(csv.DictReader(stream))
                    count = len({int(row['sid']) for row in rows})
                    last = f"s{int(rows[-1]['sid']):02d}" if rows else '-'
                else:
                    count, last = 0, '-'
                grand_total += count
                print(f'{seed:>5}  {model:<8}  {condition:<11}  {count:>2}/49  {last:>7}')
    print(f'Total: {grand_total}/882 complete')
show_progress()


In [ ]:
# seed=42, model=eeg_only, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'dirty_train', '--model_type', 'eeg_only', '--seed', '42',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=42, model=eeg_only, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'clean_train', '--model_type', 'eeg_only', '--seed', '42',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=42, model=emg_only, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'dirty_train', '--model_type', 'emg_only', '--seed', '42',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=42, model=emg_only, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'clean_train', '--model_type', 'emg_only', '--seed', '42',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=42, model=fusion, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'dirty_train', '--model_type', 'fusion', '--seed', '42',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=42, model=fusion, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'clean_train', '--model_type', 'fusion', '--seed', '42',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=1337, model=eeg_only, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'dirty_train', '--model_type', 'eeg_only', '--seed', '1337',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=1337, model=eeg_only, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'clean_train', '--model_type', 'eeg_only', '--seed', '1337',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=1337, model=emg_only, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'dirty_train', '--model_type', 'emg_only', '--seed', '1337',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=1337, model=emg_only, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'clean_train', '--model_type', 'emg_only', '--seed', '1337',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=1337, model=fusion, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'dirty_train', '--model_type', 'fusion', '--seed', '1337',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=1337, model=fusion, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'clean_train', '--model_type', 'fusion', '--seed', '1337',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=2024, model=eeg_only, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'dirty_train', '--model_type', 'eeg_only', '--seed', '2024',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=2024, model=eeg_only, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'clean_train', '--model_type', 'eeg_only', '--seed', '2024',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=2024, model=emg_only, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'dirty_train', '--model_type', 'emg_only', '--seed', '2024',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=2024, model=emg_only, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'clean_train', '--model_type', 'emg_only', '--seed', '2024',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=2024, model=fusion, condition=dirty_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'dirty_train', '--model_type', 'fusion', '--seed', '2024',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


In [ ]:
# seed=2024, model=fusion, condition=clean_train
cmd = [sys.executable, '-u', str(CODE_DIR / 'rigorous_loso.py'),
       '--condition', 'clean_train', '--model_type', 'fusion', '--seed', '2024',
       '--data_dir', str(DATA_DIR), '--exclude_json', str(EXCL_JSON),
       '--out_dir', str(OUT_DIR), '--device', 'cuda']
run_training(cmd)


## 모든 조합 완료 후 분석

분석 스크립트는 18개 CSV가 모두 49명씩 채워졌는지 검사합니다. 미완료이면 오류가 나며 결과를 만들지 않습니다.


In [ ]:
subprocess.run([sys.executable, str(CODE_DIR / 'analyze_rigorous_loso.py'),
                '--out_dir', str(OUT_DIR), '--seeds', '42', '1337', '2024'], check=True)
print('분석 결과:', OUT_DIR / 'paired_analysis.json')
